In [1]:
# Installing the crucial libraries
!apt-get update && apt-get install -y openslide-tools
!pip install openslide-python opencv-python-headless tqdm numpy matplotlib

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,389 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 http://archive.ubuntu.com/ubuntu jammy-

In [6]:
# Connecting to dropbox where the dataset is using the rclone remote
!rclone config delete dropboxdataset
!rclone config

No remotes found, make a new one?
n) New remote
s) Set configuration password
q) Quit config
n/s/q> n

Enter name for new remote.
name> dropboxdataset

Option Storage.
Type of storage to configure.
Choose a number from below, or type in your own value.
 1 / 1Fichier
   \ (fichier)
 2 / Akamai NetStorage
   \ (netstorage)
 3 / Alias for an existing remote
   \ (alias)
 4 / Amazon S3 Compliant Storage Providers including AWS, Alibaba, ArvanCloud, BizflyCloud, Ceph, ChinaMobile, Cloudflare, Cubbit, DigitalOcean, Dreamhost, Exaba, FileLu, FlashBlade, GCS, Hetzner, HuaweiOBS, IBMCOS, IDrive, Intercolo, IONOS, Leviia, Liara, Linode, LyveCloud, Magalu, Mega, Minio, Netease, Outscale, OVHcloud, Petabox, Qiniu, Rabata, RackCorp, Rclone, Scaleway, SeaweedFS, Selectel, Servercore, SpectraLogic, StackPath, Storj, Synology, TencentCOS, Wasabi, Zata, Other
   \ (s3)
 5 / Backblaze B2
   \ (b2)
 6 / Better checksums for other remotes
   \ (hasher)
 7 / Box
   \ (box)
 8 / Cache a remote
   \ (cache)


In [7]:
# Now verifying if it connected
!rclone lsd dropboxdataset:/COMP_499_Project/

          -1 2000-01-01 00:00:00        -1 CPTAC_WSI
          -1 2000-01-01 00:00:00        -1 TCGA_WSI


In [5]:
# Now installing AtlasPatch
!pip install atlas-patch
!pip install git+https://github.com/facebookresearch/sam2.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 26.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.1/106.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
  Created wheel for fairscale: filename=fairscale-0.4.13-py3-none-any.whl size=332207 sha256=0dd1240114a75b5631365562ad7fd34bad9c50daf2e27fc6dceb282f78dba3a5
  Stored in directory: /root/.cache/pip/wheels/5a/88/aa/d84b2cf1bad6b273cbf661640141a82c7b9f496e024f80aac0
Successfully built fairscale
  Cloning https://github.com/facebookresearch/sam2.git to /tmp/pip-req-build-he148lak
  Running command git clone --filter=blob:no

In [8]:
# Now setting up the paths
import os
import subprocess
import time
import glob

# Dropbox paths
# Input slides are here
TCGA_ROOT = "dropboxdataset:/COMP_499_Project/TCGA_WSI"

# Where final patch H5 files will be stored in Dropbox
PATCH_ROOT = "dropboxdataset:/COMP_499_Project/TCGA_PATCHES"

# Local temporary directory for processing
LOCAL_SLIDE_DIR = "/content/tcga_slides"
LOCAL_OUTPUT_DIR = "/content/tcga_patches"
os.makedirs(LOCAL_SLIDE_DIR, exist_ok=True)
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)

# Patch extraction parameters
PATCH_SIZE = 512          # final patch size at target magnification
TARGET_MAG = 20           # 20× magnification
FEATURE_EXTRACTORS = ["resnet50"]
BATCH_SIZE = 5

In [9]:
#Listing all the slides in Dropbox and check which have already been processed
def list_slides_in_dropbox():
    """Return list of (slide_id, remote_path) for all .svs files in TCGA_ROOT."""
    result = subprocess.run(
        ["rclone", "lsf", TCGA_ROOT, "--recursive"],
        capture_output=True, text=True
    )
    lines = result.stdout.strip().split('\n')
    slides = []
    for line in lines:
        if line.endswith('.svs'):
            parts = line.strip().split('/')
            if len(parts) >= 2:
                uuid = parts[0]
                filename = parts[-1]
                slide_id = uuid   # use UUID as unique identifier
                remote_path = f"{TCGA_ROOT}/{line.strip()}"
                slides.append((slide_id, remote_path))
    return slides

def slide_already_processed(slide_id):
    """Check if an H5 file for this slide already exists in PATCH_ROOT."""
    h5_name = f"{slide_id}.h5"
    result = subprocess.run(
        ["rclone", "lsf", f"{PATCH_ROOT}/"],
        capture_output=True, text=True
    )
    existing = result.stdout.strip().split('\n')
    return h5_name in existing

all_slides = list_slides_in_dropbox()
print(f"Total TCGA slides found: {len(all_slides)}")

to_process = []
for slide_id, path in all_slides:
    if not slide_already_processed(slide_id):
        to_process.append((slide_id, path))
    else:
        print(f"Skipping {slide_id} (already processed)")

print(f"Slides to process: {len(to_process)}")

Total TCGA slides found: 1053
Slides to process: 1053


In [ ]:
# Now we batch process using Atlas Patch
# For each batch:
#   1. Copy the SVS files from Dropbox to a local folder.
#   2. Run `atlaspatch process` on that folder, writing H5 files locally.
#   3. Upload the resulting H5 files to Dropbox.
#   4. Clean up local files.
total = len(to_process)
for i in range(0, total, BATCH_SIZE):
    batch = to_process[i:i+BATCH_SIZE]
    batch_num = i//BATCH_SIZE + 1
    total_batches = (total + BATCH_SIZE - 1)//BATCH_SIZE
    print(f"\n{'='*60}")
    print(f"Processing batch {batch_num}/{total_batches} ({len(batch)} slides)")
    print(f"{'='*60}")

    # --- Step 1: Copy slides locally ---
    local_slides = []
    for slide_id, remote_path in batch:
        local_path = os.path.join(LOCAL_SLIDE_DIR, f"{slide_id}.svs")
        print(f"  Copying {slide_id} to local...")
        !rclone copy {remote_path} {LOCAL_SLIDE_DIR} --include "*.svs" --progress
        local_slides.append(local_path)

    # --- Step 2: Run AtlasPatch on the local slides ---
    print("  Running AtlasPatch on batch...")
    # We'll process all slides in LOCAL_SLIDE_DIR (the folder containing the batch)
    # AtlasPatch will create one H5 per slide in LOCAL_OUTPUT_DIR/patches/
    !atlaspatch process {LOCAL_SLIDE_DIR} \
        --output {LOCAL_OUTPUT_DIR} \
        --patch-size {PATCH_SIZE} \
        --target-mag {TARGET_MAG} \
        --feature-extractors {','.join(FEATURE_EXTRACTORS)} \
        --device cuda \
        --fast-mode \
        --skip-existing

    # --- Step 3: Upload the H5 files to Dropbox ---
    print("  Uploading H5 files to Dropbox...")
    # The H5 files are in LOCAL_OUTPUT_DIR/patches/
    !rclone copy {LOCAL_OUTPUT_DIR}/patches/ {PATCH_ROOT}/ --progress --ignore-existing

    # --- Step 4: Clean up local files for this batch ---
    print("  Cleaning up local files...")
    !rm -rf {LOCAL_SLIDE_DIR}/* {LOCAL_OUTPUT_DIR}/patches/*

    print(f"Batch {batch_num} completed.\n")


Processing batch 1/211 (5 slides)
  Copying 01bea133-7f27-40a6-8d4e-018da17accda to local...























































































































































































































  Copying 00efe162-75c1-4451-814c-1576cf6856f8 to local...























































































































  Copying 004d8238-0a74-40bd-9547-f48a2086c416 to local...











































































  Copying 01954393-6fd5-422b-bf9e-36fab4452619 to local...







































































































































  Copying 015862cf-a450-48fa-b987-f993729436a6 to local...







































































































  Running AtlasPatch on